# JEE / NEET Automatic PDF Image & Diagram Extraction Pipeline

This pipeline automatically extracts:
1. **Original embedded image objects** directly from PDF streams.
2. **Rendered figures & diagrams** (physics diagrams, chemistry apparatus/structures, charts) detected using a pretrained DocLayNet YOLO vision model.
3. **Vector drawings & circuit diagrams** clustered directly from PDF vector graphics paths.
4. **Deduplication, blank suppression, and automatic margin trimming**.
5. **Persistent storage & incremental resume** on Google Drive with a structured JSONL manifest.

> **Manual Step Required for Google Drive Access:**
> Because the shared folder (`https://drive.google.com/drive/folders/1JsHHTtHVNjcxBeTZf5oQPO0VxtJEmLkJ`) is a shared link, open the link in your browser, click **"Add shortcut to Drive"** (or **"Organize" -> "Add shortcut"**) and select **"My Drive"**. This makes the folder immediately accessible at `/content/drive/MyDrive/<Folder_Name>` in Colab.

### Step 1: Install Dependencies
Installs PyMuPDF, Ultralytics YOLO, Hugging Face Hub, OpenCV, and Pillow.

In [ ]:
!pip install --upgrade --quiet pymupdf opencv-python pillow ultralytics huggingface_hub

### Step 2: Mount Google Drive
Mount your persistent Google Drive to load input PDFs and save extracted images.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 3: Configure Paths
Set the path where the JEE/NEET PDFs are located and where the outputs should be saved.
Excel files (`.xlsx`, `.xls`) will automatically be skipped.

In [ ]:
import os
from pathlib import Path

# Set the path to the folder shortcut in your Google Drive
# Example: '/content/drive/MyDrive/JEE_NEET_PDFs' or update to match your folder name
INPUT_DIR = "/content/drive/MyDrive/JEE_NEET_PDFs"

# Set persistent output directory in Drive
OUTPUT_DIR = "/content/drive/MyDrive/JEE_NEET_Extracted_Images"

# Parameters
RENDER_DPI = 200          # 200 DPI gives ~1654x2338 for A4, preserving fine diagram details
MODEL_NAME = "yolov8m-doclaynet.pt" # DocLayNet YOLO model (or yolov8n-doclaynet.pt / yolov8x-doclaynet.pt)
ENABLE_YOLO = True
ENABLE_DRAWINGS = True

print(f"Configured Input: {INPUT_DIR}")
print(f"Configured Output: {OUTPUT_DIR}")

### Step 4: Core Pipeline Implementation
Defines the layout detector, vector graphics parser, deduplicator, and extraction engine.

In [ ]:
import sys
import json
import hashlib
from typing import Dict, List, Tuple, Any, Optional

import pymupdf as fitz
import cv2
import numpy as np
from PIL import Image
import torch
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

def is_near_blank(img_np: np.ndarray, std_thresh: float = 8.0, white_ratio_thresh: float = 0.985) -> bool:
    """Check if an image or crop is virtually blank, uniform, or pure white."""
    if img_np is None or img_np.size == 0:
        return True
    if len(img_np.shape) == 3:
        gray = cv2.cvtColor(img_np, cv2.COLOR_BGR2GRAY)
    else:
        gray = img_np
    
    std = float(np.std(gray))
    if std < std_thresh:
        return True
    
    white_pixels = np.count_nonzero(gray > 248)
    if (white_pixels / gray.size) > white_ratio_thresh:
        return True
    return False

def trim_borders(img_np: np.ndarray, tol: int = 245, pad: int = 8) -> np.ndarray:
    """Auto-trim excessive white borders around diagrams without clipping content."""
    if img_np is None or img_np.size == 0:
        return img_np
    if len(img_np.shape) == 3:
        gray = cv2.cvtColor(img_np, cv2.COLOR_BGR2GRAY)
    else:
        gray = img_np

    mask = gray < tol
    coords = cv2.findNonZero(mask.astype(np.uint8))
    if coords is None:
        return img_np
    
    x, y, w, h = cv2.boundingRect(coords)
    H, W = img_np.shape[:2]
    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(W, x + w + pad)
    y2 = min(H, y + h + pad)

    if (x2 - x1) < 15 or (y2 - y1) < 15:
        return img_np
    return img_np[y1:y2, x1:x2]

def box_iou(b1: Tuple[float, float, float, float], b2: Tuple[float, float, float, float]) -> float:
    """Compute IoU between [x1, y1, x2, y2]."""
    x1 = max(b1[0], b2[0])
    y1 = max(b1[1], b2[1])
    x2 = min(b1[2], b2[2])
    y2 = min(b1[3], b2[3])

    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    union = a1 + a2 - inter
    if union <= 0:
        return 0.0
    return inter / union

class LayoutDetector:
    """Pretrained DocLayNet YOLO layout detector."""
    def __init__(self, model_repo: str = "hantian/yolo-doclaynet", model_filename: str = "yolov8m-doclaynet.pt"):
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        print(f"Device: {self.device} (GPU enabled: {torch.cuda.is_available()})")
        print(f"Loading DocLayNet weights ({model_filename})...")
        try:
            model_path = hf_hub_download(repo_id=model_repo, filename=model_filename)
        except Exception as e:
            print(f"Fallback to nano model: {e}")
            model_path = hf_hub_download(repo_id=model_repo, filename="yolov8n-doclaynet.pt")

        self.model = YOLO(model_path)
        self.classes = self.model.names
        print(f"Classes detected by model: {self.classes}")

        # Target Picture (6) and related visual diagram classes
        self.target_indices = [
            idx for idx, name in self.classes.items()
            if name.lower() in ["picture", "figure", "formula", "diagram"]
        ]

    def detect_figures(self, img_pil: Image.Image, conf_thresh: float = 0.25, imgsz: int = 1024) -> List[Dict[str, Any]]:
        results = self.model.predict(
            source=img_pil,
            imgsz=imgsz,
            conf=conf_thresh,
            device=self.device,
            verbose=False
        )
        detections = []
        if not results or results[0].boxes is None:
            return detections
        
        for box in results[0].boxes:
            cls_id = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            if cls_id in self.target_indices:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                detections.append({
                    "class_id": cls_id,
                    "class_name": self.classes[cls_id],
                    "conf": round(conf, 4),
                    "bbox": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]
                })
        return detections

class PDFImageExtractor:
    def __init__(
        self,
        output_dir: str,
        detector: Optional[LayoutDetector] = None,
        render_dpi: int = 200,
        min_width: int = 40,
        min_height: int = 40,
        enable_yolo: bool = True,
        enable_drawings: bool = True
    ):
        self.output_dir = Path(output_dir)
        self.detector = detector
        self.render_dpi = render_dpi
        self.min_width = min_width
        self.min_height = min_height
        self.enable_yolo = enable_yolo
        self.enable_drawings = enable_drawings

        self.embedded_dir = self.output_dir / "embedded_images"
        self.figures_dir = self.output_dir / "cropped_figures"
        self.manifest_path = self.output_dir / "extraction_manifest.jsonl"

        self.embedded_dir.mkdir(parents=True, exist_ok=True)
        self.figures_dir.mkdir(parents=True, exist_ok=True)
        self.processed_pdfs = self._load_processed_history()

    def _load_processed_history(self) -> set:
        completed = set()
        if self.manifest_path.exists():
            try:
                with open(self.manifest_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        if line.strip():
                            record = json.loads(line)
                            completed.add(record.get("pdf_name"))
            except Exception:
                pass
        return completed

    def _append_manifest(self, record: Dict[str, Any]):
        with open(self.manifest_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(record) + "\n")

    def extract_native_images(self, doc: fitz.Document, pdf_stem: str) -> List[Dict[str, Any]]:
        out_records = []
        pdf_out_dir = self.embedded_dir / pdf_stem
        seen_hashes = set()

        for page_idx in range(len(doc)):
            page = doc[page_idx]
            page_num = page_idx + 1
            image_list = page.get_images(full=True)

            for img_info in image_list:
                xref = img_info[0]
                try:
                    base_image = doc.extract_image(xref)
                except Exception:
                    continue
                if not base_image:
                    continue

                image_bytes = base_image.get("image")
                image_ext = base_image.get("ext", "png")
                width = base_image.get("width", 0)
                height = base_image.get("height", 0)

                if width < self.min_width or height < self.min_height:
                    continue

                is_full_page = (width >= 1000 and height >= 1400)

                h = hashlib.md5(image_bytes).hexdigest()
                if h in seen_hashes:
                    continue
                seen_hashes.add(h)

                try:
                    import io
                    pil_img = Image.open(io.BytesIO(image_bytes))
                    np_img = np.array(pil_img)
                    if is_near_blank(np_img):
                        continue
                except Exception:
                    continue

                pdf_out_dir.mkdir(parents=True, exist_ok=True)
                out_filename = f"{pdf_stem}_p{page_num:03d}_xref{xref}.{image_ext}"
                out_path = pdf_out_dir / out_filename

                if not out_path.exists():
                    with open(out_path, "wb") as f_out:
                        f_out.write(image_bytes)

                rec = {
                    "pdf_name": pdf_stem,
                    "page": page_num,
                    "type": "native_embedded",
                    "xref": xref,
                    "file_path": str(out_path.relative_to(self.output_dir)),
                    "dimensions": [width, height],
                    "bbox": None,
                    "is_full_page_scan": is_full_page
                }
                out_records.append(rec)
                self._append_manifest(rec)

        return out_records

    def extract_vector_drawings(self, page: fitz.Page, page_pix: fitz.Pixmap, pdf_stem: str, page_num: int) -> List[List[int]]:
        drawings = page.get_drawings()
        if not drawings:
            return []

        page_w = page.rect.width
        page_h = page.rect.height
        pix_w = page_pix.width
        pix_h = page_pix.height
        scale_x = pix_w / page_w
        scale_y = pix_h / page_h

        boxes = []
        for d in drawings:
            r = d.get("rect")
            if not r:
                continue
            if r.width >= (page_w * 0.95) and r.height < 5:
                continue
            if r.height >= (page_h * 0.95) and r.width < 5:
                continue
            if r.width < 5 and r.height < 5:
                continue
            boxes.append([r.x0, r.y0, r.x1, r.y1])

        if not boxes:
            return []

        merged = []
        margin = 35
        for b in boxes:
            placed = False
            for m in merged:
                if not (b[0] > m[2] + margin or b[2] < m[0] - margin or
                        b[1] > m[3] + margin or b[3] < m[1] - margin):
                    m[0] = min(m[0], b[0])
                    m[1] = min(m[1], b[1])
                    m[2] = max(m[2], b[2])
                    m[3] = max(m[3], b[3])
                    placed = True
                    break
            if not placed:
                merged.append(list(b))

        clusters = []
        for m in merged:
            w = m[2] - m[0]
            h = m[3] - m[1]
            if (w >= (self.min_width / scale_x) and h >= (self.min_height / scale_y) and
                w < (page_w * 0.90) and h < (page_h * 0.90)):
                px1 = max(0, int(m[0] * scale_x))
                py1 = max(0, int(m[1] * scale_y))
                px2 = min(pix_w, int(m[2] * scale_x))
                py2 = min(pix_h, int(m[3] * scale_y))
                clusters.append([px1, py1, px2, py2])

        return clusters

    def process_pdf(self, pdf_path: str, skip_if_done: bool = True) -> int:
        p_path = Path(pdf_path)
        pdf_stem = p_path.stem

        if skip_if_done and pdf_stem in self.processed_pdfs:
            print(f"⏩ Skipping already processed PDF: {pdf_stem}")
            return 0

        print(f"\n📄 Processing: {p_path.name}")
        try:
            doc = fitz.open(pdf_path)
        except Exception as e:
            print(f"❌ Error opening PDF {pdf_path}: {e}")
            return 0

        total_saved = 0

        # 1. Native embedded image extraction
        try:
            native_items = self.extract_native_images(doc, pdf_stem)
            total_saved += len(native_items)
            print(f"  ✓ Extracted {len(native_items)} native image streams")
        except Exception as e:
            print(f"  ⚠️ Native extraction warning: {e}")

        # 2. Page visual layout & vector drawings
        figures_pdf_dir = self.figures_dir / pdf_stem
        for page_idx in range(len(doc)):
            page_num = page_idx + 1
            page = doc[page_idx]

            zoom = self.render_dpi / 72.0
            mat = fitz.Matrix(zoom, zoom)
            try:
                pix = page.get_pixmap(matrix=mat, alpha=False)
            except Exception as e:
                print(f"  Failed rendering p.{page_num}: {e}")
                continue

            page_img_pil = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            page_img_np = np.frombuffer(pix.samples, dtype=np.uint8).reshape((pix.height, pix.width, 3))
            page_img_bgr = cv2.cvtColor(page_img_np, cv2.COLOR_RGB2BGR)

            candidate_boxes = []
            if self.enable_yolo and self.detector:
                try:
                    yolo_dets = self.detector.detect_figures(page_img_pil, conf_thresh=0.25, imgsz=1024)
                    for det in yolo_dets:
                        candidate_boxes.append({
                            "bbox": det["bbox"],
                            "source": f"yolo_{det['class_name']}",
                            "conf": det["conf"]
                        })
                except Exception as e:
                    print(f"  Layout detection error p.{page_num}: {e}")

            if self.enable_drawings:
                try:
                    drawing_boxes = self.extract_vector_drawings(page, pix, pdf_stem, page_num)
                    for dbox in drawing_boxes:
                        candidate_boxes.append({
                            "bbox": dbox,
                            "source": "vector_drawing_cluster",
                            "conf": 0.8
                        })
                except Exception as e:
                    print(f"  Vector clustering error p.{page_num}: {e}")

            # Deduplicate with NMS IoU
            candidate_boxes.sort(key=lambda x: x["conf"], reverse=True)
            kept_boxes = []
            for cand in candidate_boxes:
                b = cand["bbox"]
                w = b[2] - b[0]
                h = b[3] - b[1]
                if w >= (pix.width * 0.92) and h >= (pix.height * 0.92):
                    continue
                if w < self.min_width or h < self.min_height:
                    continue

                overlap = False
                for k in kept_boxes:
                    if box_iou(b, k["bbox"]) > 0.45:
                        overlap = True
                        break
                if not overlap:
                    kept_boxes.append(cand)

            for crop_idx, crop_info in enumerate(kept_boxes):
                bx1, by1, bx2, by2 = [int(v) for v in crop_info["bbox"]]
                bx1 = max(0, bx1 - 6)
                by1 = max(0, by1 - 6)
                bx2 = min(pix.width, bx2 + 6)
                by2 = min(pix.height, by2 + 6)

                crop_np = page_img_bgr[by1:by2, bx1:bx2]
                if crop_np.size == 0:
                    continue

                trimmed_np = trim_borders(crop_np)
                if is_near_blank(trimmed_np):
                    continue

                figures_pdf_dir.mkdir(parents=True, exist_ok=True)
                out_filename = f"{pdf_stem}_p{page_num:03d}_fig{crop_idx + 1:02d}.png"
                out_path = figures_pdf_dir / out_filename

                cv2.imwrite(str(out_path), trimmed_np)
                total_saved += 1

                rec = {
                    "pdf_name": pdf_stem,
                    "page": page_num,
                    "type": "rendered_crop",
                    "source": crop_info["source"],
                    "confidence": crop_info["conf"],
                    "file_path": str(out_path.relative_to(self.output_dir)),
                    "dimensions": [int(trimmed_np.shape[1]), int(trimmed_np.shape[0])],
                    "bbox": [bx1, by1, bx2, by2]
                }
                self._append_manifest(rec)

        doc.close()
        self.processed_pdfs.add(pdf_stem)
        print(f"  ✓ Total extracted from {pdf_stem}: {total_saved} images/diagrams")
        return total_saved

### Step 5: Execute Full Pipeline
Finds all PDFs recursively, ignores Excel files, runs extraction, and saves to Google Drive.

In [ ]:
input_path = Path(INPUT_DIR)
if not input_path.exists():
    raise FileNotFoundError(f"Input directory not found: {INPUT_DIR}. Did you add the folder shortcut to Google Drive?")

# Search recursively for PDFs and ignore Excel files
pdf_files = sorted(list(input_path.glob("**/*.pdf")) + list(input_path.glob("**/*.PDF")))
print(f"Found {len(pdf_files)} PDF files to process in {INPUT_DIR}\n")

# Initialize detector
detector = None
if ENABLE_YOLO:
    detector = LayoutDetector(model_filename=MODEL_NAME)

extractor = PDFImageExtractor(
    output_dir=OUTPUT_DIR,
    detector=detector,
    render_dpi=RENDER_DPI,
    enable_yolo=(detector is not None),
    enable_drawings=ENABLE_DRAWINGS
)

total_all = 0
for i, pdf_file in enumerate(pdf_files):
    print(f"[{i+1}/{len(pdf_files)}] {pdf_file.name}")
    count = extractor.process_pdf(str(pdf_file))
    total_all += count

print("\n" + "="*60)
print(f"🎉 Completed! Total images & figures extracted: {total_all}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Manifest location: {extractor.manifest_path}")
print("="*60)

### Step 6: Preview Sample Extracted Figures
Quickly inspect sample extracted figures and the manifest directly inside Colab.

In [ ]:
import matplotlib.pyplot as plt

manifest_file = Path(OUTPUT_DIR) / "extraction_manifest.jsonl"
if manifest_file.exists():
    records = []
    with open(manifest_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    print(f"Total manifest records logged: {len(records)}")

    # Preview first 4 figures if available
    sample_images = [r for r in records if r.get("type") == "rendered_crop"][:4]
    if sample_images:
        fig, axes = plt.subplots(1, len(sample_images), figsize=(16, 4))
        if len(sample_images) == 1:
            axes = [axes]
        for ax, rec in zip(axes, sample_images):
            img_path = Path(OUTPUT_DIR) / rec["file_path"]
            if img_path.exists():
                img = Image.open(img_path)
                ax.imshow(img)
                ax.set_title(f"{rec['pdf_name'][:12]}.. p.{rec['page']}", fontsize=10)
                ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print("No manifest records yet.")

---
# Stage 2: Red-Box Labeled Reference PDF Diagnostic & Ground-Truth Extraction

This stage processes the reference pair:
- **Original PDF**: `Copy of original (4).pdf`
- **Red-box-labeled PDF**: `Copy of original (4) red boxes marked.pdf`

It diagnoses whether red boxes are PDF annotations, vector drawings, or flattened pixels, recovers the coordinates, excludes the red outlines, crops clean diagrams directly from the original PDF, displays a side-by-side inspection preview, and exports standard YOLO dataset annotations.

### Step 7: Configure Reference PDF Paths
Point to the exact location of `Copy of original (4).pdf` and `Copy of original (4) red boxes marked.pdf`.

In [ ]:
import os
import sys
from pathlib import Path

# ----------------------------------------------------------------------------------
# STEP 7: SELF-CONTAINED CONFIGURATION & VERIFICATION
# ----------------------------------------------------------------------------------
# Adjust these paths to the exact location of your reference PDFs.
# Common locations in Google Colab:
#   '/content/drive/MyDrive/Copy of original (4).pdf'
#   '/content/drive/MyDrive/JEE_NEET_PDFs/Copy of original (4).pdf'
# ----------------------------------------------------------------------------------
ORIGINAL_PDF_PATH = Path('/content/drive/MyDrive/Copy of original (4).pdf')
MARKED_PDF_PATH = Path('/content/drive/MyDrive/Copy of original (4) red boxes marked.pdf')

# Auto-detect in common subfolder if not directly in root MyDrive
if not ORIGINAL_PDF_PATH.exists():
    cand = Path('/content/drive/MyDrive/JEE_NEET_PDFs/Copy of original (4).pdf')
    if cand.exists():
        ORIGINAL_PDF_PATH = cand

if not MARKED_PDF_PATH.exists():
    cand = Path('/content/drive/MyDrive/JEE_NEET_PDFs/Copy of original (4) red boxes marked.pdf')
    if cand.exists():
        MARKED_PDF_PATH = cand

# Destination directories (Preserves recovered_crops and recovered_crops_v2 intact)
STAGE2_OUTPUT_DIR = Path('/content/drive/MyDrive/JEE_NEET_Extracted_Images/stage2_ground_truth')
STAGE2_CROPS_DIR = STAGE2_OUTPUT_DIR / 'recovered_crops_v3'
YOLO_DATASET_DIR = STAGE2_OUTPUT_DIR / 'yolo_dataset'

STAGE2_CROPS_DIR.mkdir(parents=True, exist_ok=True)
YOLO_DATASET_DIR.mkdir(parents=True, exist_ok=True)

orig_ok = ORIGINAL_PDF_PATH.exists()
mark_ok = MARKED_PDF_PATH.exists()

print('='*70)
print('STAGE 2 REFERENCE PDF VERIFICATION')
print('='*70)
print(f'Original PDF:  {ORIGINAL_PDF_PATH} -> [{"EXISTS" if orig_ok else "MISSING"}]')
print(f'Marked PDF:    {MARKED_PDF_PATH} -> [{"EXISTS" if mark_ok else "MISSING"}]')
print(f'Target Output: {STAGE2_CROPS_DIR}')
print('='*70)

if not orig_ok or not mark_ok:
    print('\n❌ DIAGNOSTIC NOTICE:')
    print('One or both reference PDFs could not be found at the paths specified.')
    print('Please edit ORIGINAL_PDF_PATH and MARKED_PDF_PATH in this cell to point to your files in Google Drive.')
    raise FileNotFoundError('Reference PDF paths do not exist. Please update the path variables above.')
else:
    print('\n✅ Both reference PDFs verified! Proceed to Step 8.')


### Step 8: Diagnostic Cell (PDF Annotations vs Vector Drawings vs Flattened Pixels)
Inspects the marked PDF using PyMuPDF to automatically detect how the red bounding boxes are encoded.

In [ ]:
import pymupdf as fitz
from pathlib import Path

def diagnose_marked_pdf(pdf_path, sample_pages=5):
    """Inspects marked PDF to identify whether red boxes are annotations, vector drawings, or raster pixels."""
    doc = fitz.open(str(pdf_path))
    total_pages = len(doc)
    print(f'Diagnosing marked PDF: {pdf_path} ({total_pages} total pages)\n')
    
    red_annots = []
    red_drawings = []
    has_images = False
    
    for p_idx in range(min(total_pages, sample_pages)):
        p = doc[p_idx]
        # Check PDF Annotations
        for a in p.annots():
            stroke = a.colors.get('stroke') if a.colors else None
            is_red = False
            if stroke and len(stroke) >= 3 and stroke[0] > 0.6 and stroke[1] < 0.4 and stroke[2] < 0.4:
                is_red = True
            red_annots.append({'page': p_idx + 1, 'type': a.type[1], 'rect': list(a.rect), 'is_red': is_red})
            
        # Check Vector Drawings in PDF graphics stream
        for d in p.get_drawings():
            color = d.get('color') or d.get('fill')
            if color and len(color) >= 3 and color[0] > 0.7 and color[1] < 0.35 and color[2] < 0.35:
                r = d.get('rect')
                if r and r.width > 20 and r.height > 20:
                    red_drawings.append({'page': p_idx + 1, 'rect': [r.x0, r.y0, r.x1, r.y1]})
        
        if len(p.get_images()) > 0:
            has_images = True
            
    print('--- DIAGNOSTIC RESULTS ---')
    print(f'• PDF Annotations found:     {len(red_annots)}')
    print(f'• Red Vector Drawings found: {len(red_drawings)}')
    print(f'• Raster Images present:     {has_images}')
    
    nature = 'flattened_pixels'
    if red_annots:
        nature = 'pdf_annotations'
        print('==> Red boxes detected as: PDF ANNOTATIONS')
    elif red_drawings:
        nature = 'vector_drawings'
        print('==> Red boxes detected as: VECTOR DRAWINGS')
    else:
        nature = 'flattened_pixels'
        print('==> Red boxes detected as: FLATTENED RASTER PIXELS (Using enhanced OpenCV HSV contour detector)')
    
    doc.close()
    return nature

box_nature = diagnose_marked_pdf(MARKED_PDF_PATH)


### Step 9: Recover Bounding Boxes & Extract Clean Crops (Excluding Red Border)
Matches marked pages to the original PDF, recovers the red box coordinates, applies inward border trimming (excluding the red outline and outer text), and extracts clean diagrams directly from the original PDF.

In [ ]:
import cv2
import numpy as np
import pymupdf as fitz
from pathlib import Path

def box_iou(b1, b2):
    """Intersection over Union (IoU) between [x1, y1, x2, y2]."""
    x1 = max(b1[0], b2[0])
    y1 = max(b1[1], b2[1])
    x2 = min(b1[2], b2[2])
    y2 = min(b1[3], b2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    a1 = max(0.0, b1[2] - b1[0]) * max(0.0, b1[3] - b1[1])
    a2 = max(0.0, b2[2] - b2[0]) * max(0.0, b2[3] - b2[1])
    union = a1 + a2 - inter
    return 0.0 if union <= 0 else inter / union

def is_nested(inner, outer, tol=12):
    """Check if inner box is completely or mostly enclosed within outer box."""
    ix1, iy1, ix2, iy2 = inner
    ox1, oy1, ox2, oy2 = outer
    return (ix1 >= ox1 - tol and iy1 >= oy1 - tol and ix2 <= ox2 + tol and iy2 <= oy2 + tol)

def detect_red_boxes_opencv(page_img_bgr, min_w=60, min_h=50, border_inward_trim=5, iou_thresh=0.35, require_frame_structure=True):
    """Robust red bounding-box detector: multi-wrap HSV, framing perimeter test, nested suppression, and NMS."""
    if page_img_bgr is None or page_img_bgr.size == 0:
        return []
    H, W = page_img_bgr.shape[:2]
    hsv = cv2.cvtColor(page_img_bgr, cv2.COLOR_BGR2HSV)
    
    # Dual HSV red thresholds
    lower_red1 = np.array([0, 90, 80])
    upper_red1 = np.array([12, 255, 255])
    lower_red2 = np.array([165, 90, 80])
    upper_red2 = np.array([180, 255, 255])
    mask = cv2.bitwise_or(cv2.inRange(hsv, lower_red1, upper_red1), cv2.inRange(hsv, lower_red2, upper_red2))
    
    clean_k = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    cleaned = cv2.morphologyEx(mask, cv2.MORPH_OPEN, clean_k)
    bridge_k = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    closed = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, bridge_k, iterations=2)
    
    contours, hierarchy = cv2.findContours(closed, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    if not contours or hierarchy is None:
        return []
    
    candidate_boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w < min_w or h < min_h or w > (W * 0.95) or h > (H * 0.95):
            continue
        aspect = w / float(h)
        if aspect < 0.12 or aspect > 8.5:
            continue
            
        # Frame structure verification (rejects solid cartoon fills / non-frame elements)
        if require_frame_structure:
            bw = max(2, int(min(w, h) * 0.12))
            top_s = cleaned[y : y + bw, x : x + w]
            bot_s = cleaned[max(0, y + h - bw) : y + h, x : x + w]
            left_s = cleaned[y : y + h, x : x + bw]
            right_s = cleaned[y : y + h, max(0, x + w - bw) : x + w]
            border_red = cv2.countNonZero(top_s) + cv2.countNonZero(bot_s) + cv2.countNonZero(left_s) + cv2.countNonZero(right_s)
            
            mx, my = int(w * 0.20), int(h * 0.20)
            interior = cleaned[y + my : y + h - my, x + mx : x + w - mx]
            interior_fill = cv2.countNonZero(interior) / float(interior.size) if interior.size > 0 else 0.0
            if interior_fill > 0.40:
                continue  # Discard solid/blob fills
            if border_red < ((2 * w + 2 * h) * 1.5 * 0.25):
                continue
                
        candidate_boxes.append([x, y, x + w, y + h, w * h])
        
    if not candidate_boxes:
        return []
    
    # Sort by area descending
    candidate_boxes.sort(key=lambda b: b[4], reverse=True)
    
    # Suppress nested internal boxes (retaining outer diagram frame)
    filtered_enclosing = []
    for cand in candidate_boxes:
        c_box = cand[:4]
        nested = False
        for kept in filtered_enclosing:
            if is_nested(c_box, kept[:4], tol=12):
                nested = True
                break
        if not nested:
            filtered_enclosing.append(cand)
            
    # NMS deduplication
    final_boxes = []
    for cand in filtered_enclosing:
        b = cand[:4]
        overlap = False
        for k in final_boxes:
            if box_iou(b, k) > iou_thresh:
                overlap = True
                break
        if not overlap:
            final_boxes.append(b)
            
    # Inward trimming to exclude red outline
    trimmed = []
    for (x1, y1, x2, y2) in final_boxes:
        tx1 = min(W, x1 + border_inward_trim)
        ty1 = min(H, y1 + border_inward_trim)
        tx2 = max(0, x2 - border_inward_trim)
        ty2 = max(0, y2 - border_inward_trim)
        if (tx2 - tx1) >= (min_w - 2 * border_inward_trim) and (ty2 - ty1) >= (min_h - 2 * border_inward_trim):
            trimmed.append((tx1, ty1, tx2, ty2))
            
    trimmed.sort(key=lambda b: (b[1], b[0]))
    return trimmed

# Execute recovery from original clean PDF using marked PDF coordinates
doc_orig = fitz.open(str(ORIGINAL_PDF_PATH))
doc_mark = fitz.open(str(MARKED_PDF_PATH))

total_pages = min(len(doc_orig), len(doc_mark))
RENDER_DPI = 200
zoom = RENDER_DPI / 72.0
mat = fitz.Matrix(zoom, zoom)
BORDER_INWARD_TRIM = 5

ground_truth_records = []
print(f'Processing {total_pages} pages for ground-truth extraction...\n')

for p_idx in range(total_pages):
    p_num = p_idx + 1
    page_m = doc_mark[p_idx]
    page_o = doc_orig[p_idx]
    
    pix_m = page_m.get_pixmap(matrix=mat, alpha=False)
    img_m_bgr = cv2.cvtColor(np.frombuffer(pix_m.samples, dtype=np.uint8).reshape((pix_m.height, pix_m.width, 3)), cv2.COLOR_RGB2BGR)
    
    pix_o = page_o.get_pixmap(matrix=mat, alpha=False)
    img_o_bgr = cv2.cvtColor(np.frombuffer(pix_o.samples, dtype=np.uint8).reshape((pix_o.height, pix_o.width, 3)), cv2.COLOR_RGB2BGR)
    
    boxes = []
    scale_x = pix_m.width / page_m.rect.width
    scale_y = pix_m.height / page_m.rect.height
    
    if box_nature == 'vector_drawings':
        for d in page_m.get_drawings():
            color = d.get('color') or d.get('fill')
            if color and len(color) >= 3 and color[0] > 0.7 and color[1] < 0.35 and color[2] < 0.35:
                r = d.get('rect')
                if r and r.width > 30 and r.height > 30:
                    boxes.append((
                        int(r.x0 * scale_x) + BORDER_INWARD_TRIM,
                        int(r.y0 * scale_y) + BORDER_INWARD_TRIM,
                        int(r.x1 * scale_x) - BORDER_INWARD_TRIM,
                        int(r.y1 * scale_y) - BORDER_INWARD_TRIM
                    ))
    elif box_nature == 'pdf_annotations':
        for a in page_m.annots():
            r = a.rect
            boxes.append((
                int(r.x0 * scale_x) + BORDER_INWARD_TRIM,
                int(r.y0 * scale_y) + BORDER_INWARD_TRIM,
                int(r.x1 * scale_x) - BORDER_INWARD_TRIM,
                int(r.y1 * scale_y) - BORDER_INWARD_TRIM
            ))
            
    if not boxes:
        boxes = detect_red_boxes_opencv(img_m_bgr, border_inward_trim=BORDER_INWARD_TRIM)
        
    for crop_idx, (x1, y1, x2, y2) in enumerate(boxes):
        crop = img_o_bgr[y1:y2, x1:x2]
        if crop.size == 0 or crop.shape[0] < 20 or crop.shape[1] < 20:
            continue
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        if np.std(gray) < 6.0:
            continue
            
        crop_w = int(x2 - x1)
        crop_h = int(y2 - y1)
        aspect = round(crop_w / float(crop_h), 2)
        
        # Quality flag check
        is_suspicious = False
        flag_reasons = []
        if crop_w < 70 or crop_h < 60:
            is_suspicious = True
            flag_reasons.append('Very small crop')
        if aspect > 5.0 or aspect < 0.2:
            is_suspicious = True
            flag_reasons.append(f'Extreme aspect ({aspect})')
            
        crop_name = f'orig4_p{p_num:03d}_crop{crop_idx + 1:02d}.png'
        crop_path = STAGE2_CROPS_DIR / crop_name
        cv2.imwrite(str(crop_path), crop)
        
        rec = {
            'page': p_num,
            'crop_idx': crop_idx + 1,
            'crop_filename': crop_name,
            'crop_path': str(crop_path),
            'bbox': [x1, y1, x2, y2],
            'dimensions': [crop_w, crop_h],
            'aspect_ratio': aspect,
            'page_dim': [pix_o.width, pix_o.height],
            'is_suspicious': is_suspicious,
            'flag_reasons': flag_reasons
        }
        ground_truth_records.append(rec)
        print(f'  [p.{p_num:02d}] Crop {crop_idx+1}: {crop_w}x{crop_h}px (Aspect: {aspect}) -> BBox: [{x1}, {y1}, {x2}, {y2}]' + (f' ⚠️ Flags: {flag_reasons}' if is_suspicious else ''))

doc_orig.close()
doc_mark.close()

print(f'\n✅ Successfully recovered {len(ground_truth_records)} valid target diagram crops.')
print(f'Crops saved in: {STAGE2_CROPS_DIR}')
if ground_truth_records:
    pages_with_crops = sorted(list(set(r["page"] for r in ground_truth_records)))
    print(f'Pages with diagrams: {pages_with_crops}')


### Step 10: Inspection Preview (Source Page Region vs Clean Crop)
Side-by-side comparison of the marked region on the marked page vs the clean extracted crop from the original PDF to verify exclusion of red outlines and surrounding text.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import pymupdf as fitz
import numpy as np

if ground_truth_records:
    doc_m = fitz.open(str(MARKED_PDF_PATH))
    num_show = len(ground_truth_records)  # Show all recovered crops for complete verification
    print(f'Displaying comprehensive inspection preview for all {num_show} recovered crops:\n')
    
    fig, axes = plt.subplots(num_show, 2, figsize=(14, 4.3 * num_show))
    if num_show == 1:
        axes = [axes]
        
    for i in range(num_show):
        rec = ground_truth_records[i]
        p_num = rec['page']
        x1, y1, x2, y2 = rec['bbox']
        w, h = rec['dimensions']
        aspect = rec['aspect_ratio']
        flags = rec.get('flag_reasons', [])
        
        p_m = doc_m[p_num - 1]
        pix_m = p_m.get_pixmap(matrix=mat, alpha=False)
        img_m_np = np.frombuffer(pix_m.samples, dtype=np.uint8).reshape((pix_m.height, pix_m.width, 3))
        
        # Context region with 35px boundary
        cy1, cy2 = max(0, y1 - 35), min(pix_m.height, y2 + 35)
        cx1, cx2 = max(0, x1 - 35), min(pix_m.width, x2 + 35)
        marked_context = img_m_np[cy1:cy2, cx1:cx2]
        
        clean_crop = Image.open(rec['crop_path'])
        
        axes[i][0].imshow(marked_context)
        title_left = f'Page {p_num} (Marked Context) - Target Frame: [{x1}, {y1}, {x2}, {y2}]'
        axes[i][0].set_title(title_left, fontsize=10, fontweight='bold')
        axes[i][0].axis('off')
        
        axes[i][1].imshow(clean_crop)
        title_right = f'Clean Crop: {rec["crop_filename"]} ({w}x{h}px, Aspect: {aspect})'
        if flags:
            title_right += f' ⚠️ Flags: {flags}'
            axes[i][1].set_title(title_right, fontsize=10, fontweight='bold', color='crimson')
        else:
            axes[i][1].set_title(title_right, fontsize=10, fontweight='bold', color='darkgreen')
        axes[i][1].axis('off')
        
    doc_m.close()
    plt.tight_layout()
    plt.show()
else:
    print('No records found to preview. Please verify Step 7 and Step 9.')


### Step 11: Export YOLO-Format Training Dataset (DO NOT TRAIN YET)
Converts the verified ground-truth bounding boxes into YOLO label format (`<class_id> <x_center> <y_center> <width> <height>` normalized to 0.0 - 1.0) and generates `data.yaml`.

> **SAFETY GATE**: Do NOT start model training or full dataset extraction until you have visually reviewed and approved the previewed crops in Step 10.

In [ ]:
# Group records by page
page_groups = {}
for r in ground_truth_records:
    p = r["page"]
    page_groups.setdefault(p, []).append(r)

images_dir = YOLO_DATASET_DIR / "images"
labels_dir = YOLO_DATASET_DIR / "labels"
images_dir.mkdir(parents=True, exist_ok=True)
labels_dir.mkdir(parents=True, exist_ok=True)

doc_orig = fitz.open(str(ORIGINAL_PDF_PATH))

for p_num, recs in page_groups.items():
    page = doc_orig[p_num - 1]
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img_bgr = cv2.cvtColor(np.frombuffer(pix.samples, dtype=np.uint8).reshape((pix.height, pix.width, 3)), cv2.COLOR_RGB2BGR)
    W, H = pix.width, pix.height
    
    img_stem = f"orig4_p{p_num:03d}"
    cv2.imwrite(str(images_dir / f"{img_stem}.png"), img_bgr)
    
    with open(labels_dir / f"{img_stem}.txt", "w", encoding="utf-8") as f_lbl:
        for r in recs:
            x1, y1, x2, y2 = r["bbox"]
            x_center = (x1 + x2) / 2.0 / W
            y_center = (y1 + y2) / 2.0 / H
            norm_w = (x2 - x1) / W
            norm_h = (y2 - y1) / H
            f_lbl.write(f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")

doc_orig.close()

data_yaml_content = f"""# YOLO Dataset Configuration for JEE/NEET Figures
path: {YOLO_DATASET_DIR.resolve().as_posix()}
train: images
val: images

nc: 1
names: ['figure']
"""

with open(YOLO_DATASET_DIR / "data.yaml", "w", encoding="utf-8") as f_y:
    f_y.write(data_yaml_content)

print(f"\n✅ YOLO Dataset ready for review:")
print(f"  Images: {len(page_groups)} page images saved in {images_dir}")
print(f"  Labels: {len(ground_truth_records)} bounding box annotations saved in {labels_dir}")
print(f"  Config: {YOLO_DATASET_DIR / 'data.yaml'}")
print("\nDataset is prepared. Review crops and label counts before proceeding with any model training.")